# recon3d: framing statistics + demo examples (Kaggle, CPU is enough)

Measures how large and where the object sits in the training silhouettes, so the web app can frame
uploads the same way, and exports a few example silhouettes for the demo.

**Settings:** Accelerator **None** (CPU), Internet **on**.
**Inputs:** datasets `recon3d-shapenet6-v2` and `recon3d-unseen6-v1`.
Output to download: `framing/framing_bundle.zip` (a few hundred KB). Runs in about 2-5 minutes.

In [ ]:
import os, sys, glob, json, subprocess
E = os.environ.get
GITHUB_REPO = "https://github.com/Punithb2/recon3d"
PIP_SPEC   = E("R3D_PIP_SPEC", f"git+{GITHUB_REPO}.git@main")
INPUT_ROOT = E("R3D_INPUT", "/kaggle/input")
OUT        = os.path.join(E("R3D_WORK", "/kaggle/working"), "framing")

r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", PIP_SPEC], capture_output=True, text=True)
print(r.stderr[-1000:]); assert r.returncode == 0, "pip install failed"

metas = [(os.path.dirname(p), json.load(open(p))) for p in glob.glob(f"{INPUT_ROOT}/**/meta.json", recursive=True)]
seen   = [d for d, m in metas if m.get("role") != "unseen_categories" and m["counts"].get("train", 0) > 0]
unseen = [d for d, m in metas if m.get("role") == "unseen_categories"]
assert len(seen) == 1 and len(unseen) == 1, f"attach exactly one training and one unseen dataset: {seen} {unseen}"
print("seen:", seen[0], "\nunseen:", unseen[0])

In [ ]:
p = subprocess.run(["recon3d", "framing", "--data", seen[0], "--unseen", unseen[0], "--out", OUT],
                   capture_output=True, text=True)
print(p.stdout, p.stderr[-2000:])
assert p.returncode == 0
stats = json.load(open(os.path.join(OUT, "framing.json")))
print(json.dumps({k: stats[k] for k in ("images", "fill", "center_x", "center_y", "fill_by_category",
                                        "fill_by_elevation", "recommended_fill")}, indent=1))

In [ ]:
from IPython.display import Image as IPImage, display
import matplotlib.pyplot as plt
from PIL import Image
files = sorted(glob.glob(os.path.join(OUT, "examples", "*.png")))
fig, axes = plt.subplots(2, (len(files) + 1) // 2, figsize=(2 * ((len(files) + 1) // 2), 4.4))
for ax, f in zip(axes.ravel(), files):
    ax.imshow(Image.open(f), cmap="gray"); ax.set_title(os.path.basename(f)[:-4], fontsize=7); ax.axis("off")
for ax in axes.ravel()[len(files):]:
    ax.axis("off")
plt.tight_layout(); plt.show()
print("Download from Output: framing/framing_bundle.zip")